# DSS-LVR nonlinear playground — initialization fix
The simulator and target ELBO are unchanged. `slab_init="auto"` preserves the original shallow initialization and scales only the initial slab standard deviations in deeper networks. Use `slab_init="legacy"` to reproduce the old initialization.
The printed `diagR2` uses a subset of the training observations; it is not validation R2. Final metrics use the held-out test set.


In [37]:
from pathlib import Path
import math
import random
import sys
import time

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score


def find_project_root():
    here = Path.cwd().resolve()
    for root in (here, *here.parents):
        if (root / "Python" / "model2.py").exists() and (root / "Python" / "bnn_metric.py").exists():
            return root
    raise FileNotFoundError("Run this notebook inside the NFlow repository.")

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

import Python.model2 as md
import Python.bnn_metric as metric

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32

print("project root:", ROOT)
print("device      :", DEVICE)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
project root: E:\positron\NFlow
device      : cpu


## Simulation
The supplied mixed/axis ReLU simulator is retained exactly. Interaction and quadratic terms are controlled independently.


In [38]:
def simfun(
    n=600,
    p=100,
    n_active=10,
    n_true_units=10,
    activation="relu",          # "none" or "relu"
    structure="mixed",          # "axis" or "mixed"
    features_per_unit=3,        # only used for mixed
    n_interactions=0,           # 0, 1, 2, 3
    n_quadratic=0,              # 0, 1, 2, 3
    extra_scale=0.5,
    sigma2=1.0,
    target_signal_sd=1.5,
    x_low=-2.5,
    x_high=2.5,
    seed=123,
    device=None,
    dtype=torch.float32,
):
    device = torch.device("cpu") if device is None else torch.device(device)
    rng = np.random.default_rng(seed)
    gen = torch.Generator(device=device)
    gen.manual_seed(seed)

    n = int(n)
    p = int(p)
    n_active = int(n_active)
    K = int(n_true_units)
    n_interactions = int(n_interactions)
    n_quadratic = int(n_quadratic)

    if activation not in {"none", "relu"}:
        raise ValueError("activation must be 'none' or 'relu'.")
    if structure not in {"axis", "mixed"}:
        raise ValueError("structure must be 'axis' or 'mixed'.")
    if not 0 <= n_interactions <= 3 or not 0 <= n_quadratic <= 3:
        raise ValueError("n_interactions and n_quadratic must be between 0 and 3.")

    X = (
        float(x_low)
        + (float(x_high) - float(x_low))
        * torch.rand(n, p, generator=gen, device=device, dtype=dtype)
    )

    active_idx = np.sort(rng.choice(p, size=n_active, replace=False))
    active_t = torch.as_tensor(active_idx, device=device, dtype=torch.long)
    Xa = X.index_select(1, active_t)

    W = np.zeros((K, n_active), dtype=np.float32)

    if structure == "axis":
        if K < n_active:
            raise ValueError("axis requires n_true_units >= n_active.")

        supports = [
            {unit % n_active}
            for unit in range(K)
        ]

    else:
        m = min(int(features_per_unit), n_active)

        if K * m < n_active:
            raise ValueError(
                "mixed requires n_true_units * features_per_unit >= n_active."
            )

        supports = [set() for _ in range(K)]

        for pos, feature in enumerate(rng.permutation(n_active)):
            supports[pos % K].add(int(feature))

        for unit in range(K):
            while len(supports[unit]) < m:
                supports[unit].add(int(rng.integers(n_active)))

    for unit, support in enumerate(supports):
        idx = np.asarray(sorted(support), dtype=int)

        signs = rng.choice([-1.0, 1.0], size=len(idx))
        magnitude = rng.uniform(0.8, 1.2)

        W[unit, idx] = (
            magnitude
            * signs
            / np.sqrt(len(idx))
        )

    if activation == "relu":
        bias = rng.uniform(-0.8, 0.8, size=K).astype(np.float32)
    else:
        bias = np.zeros(K, dtype=np.float32)

    # Positive amplitudes avoid accidental cancellation of declared
    # active predictors while still allowing mixed signs through W.
    amplitude = rng.uniform(0.8, 1.2, size=K).astype(np.float32)

    Wt = torch.as_tensor(W, device=device, dtype=dtype)
    bt = torch.as_tensor(bias, device=device, dtype=dtype)
    at = torch.as_tensor(amplitude, device=device, dtype=dtype)

    pre = Xa @ Wt.T - bt

    if activation == "relu":
        hidden = F.relu(pre)
    else:
        hidden = pre

    signal = hidden @ at

    interaction_pairs = []
    if n_interactions > 0:
        candidates = [
            (j, k)
            for j in range(n_active)
            for k in range(j + 1, n_active)
        ]
        rng.shuffle(candidates)
        interaction_pairs = candidates[:n_interactions]

        for j, k in interaction_pairs:
            term = Xa[:, j] * Xa[:, k]
            term = (
                term - term.mean()
            ) / term.std(unbiased=False).clamp_min(1e-8)

            signal = signal + float(extra_scale) * term

    quadratic_features = []
    if n_quadratic > 0:
        quadratic_features = rng.choice(
            n_active,
            size=min(n_quadratic, n_active),
            replace=False,
        ).tolist()

        for j in quadratic_features:
            term = Xa[:, j].square()
            term = (
                term - term.mean()
            ) / term.std(unbiased=False).clamp_min(1e-8)

            signal = signal + float(extra_scale) * term

    signal = signal - signal.mean()
    signal = (
        signal
        * float(target_signal_sd)
        / signal.std(unbiased=False).clamp_min(1e-8)
    )

    y = signal + math.sqrt(float(sigma2)) * torch.randn(
        n,
        generator=gen,
        device=device,
        dtype=dtype,
    )

    feature_true = torch.zeros(
        p,
        device=device,
        dtype=dtype,
    )
    feature_true[active_t] = 1.0

    info = {
        "sim": f"{structure}_{activation}",
        "seed": int(seed),
        "n": n,
        "p": p,
        "n_active": n_active,
        "active_idx": active_idx,
        "feature_true": feature_true.detach().cpu().numpy(),
        "n_true_units": K,
        "activation": activation,
        "structure": structure,
        "features_per_unit": (
            1 if structure == "axis"
            else int(features_per_unit)
        ),
        "teacher_supports": [
            active_idx[np.asarray(sorted(s), dtype=int)].tolist()
            for s in supports
        ],
        "interaction_pairs": [
            (int(active_idx[j]), int(active_idx[k]))
            for j, k in interaction_pairs
        ],
        "quadratic_features": [
            int(active_idx[j])
            for j in quadratic_features
        ],
        "n_interactions": len(interaction_pairs),
        "n_quadratic": len(quadratic_features),
        "sigma2": float(sigma2),
        "signal_sd": float(signal.std(unbiased=False)),
    }

    return X, y, feature_true, signal, info

## Minimal trainer

This is the only training path used below. There is no `history` object and no post-training inverse/sanity pass. `R_final` is sampled once.

In [39]:
def train_fast(
    X_train, y_train, X_eval, signal_eval, X_test, signal_test, *, truth,
    selection_mode="feature_group", hidden_dims=(20,), sigma2=1.0,
    init_sd=0.5, K_flow=4, flow_hidden_units=128, flow_hidden_layers=2,
    scale_clip=2.0, flow_seed=123, iaf_ordering_scheme="cyclic3",
    iaf_shuffle_within_role=True, gate_type="normalized_requ", gate_scale=1.0,
    epochs=1200, warmup_epochs=300, lr=3e-4, R_train=32, R_eval=128,
    R_final=500, eval_every=300, init_loc_jitter=0.05, grad_clip=5.0,
    support_threshold=0.5, seed=123,
    slab_init="auto", slab_sd_ratio=0.1, slab_bias_sd=0.02,
):
    random.seed(int(seed)); np.random.seed(int(seed)); torch.manual_seed(int(seed))
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(int(seed))

    model = md.GroupedBNNVI(
        X=X_train, y=y_train, input_dim=X_train.shape[1], hidden_dims=tuple(hidden_dims), out_dim=1,
        selection_mode=selection_mode, family="gaussian", sigma2=float(sigma2),
        init_sd=float(init_sd), K_flow=int(K_flow), flow_type="iaf",
        flow_hidden_units=int(flow_hidden_units), flow_hidden_layers=int(flow_hidden_layers),
        scale_clip=float(scale_clip), flow_seed=int(flow_seed),
        iaf_ordering_scheme=iaf_ordering_scheme, iaf_shuffle_within_role=bool(iaf_shuffle_within_role),
        gate_type=gate_type, gate_scale=float(gate_scale),
        slab_init=slab_init, slab_sd_ratio=slab_sd_ratio, slab_bias_sd=slab_bias_sd,
    ).to(DEVICE)

    if init_loc_jitter > 0:
        with torch.no_grad():
            model.q0.loc.add_(float(init_loc_jitter) * torch.randn_like(model.q0.loc))

    optimizer = torch.optim.Adam(model.parameters(), lr=float(lr))
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    started = time.perf_counter()

    for epoch in range(1, int(epochs) + 1):
        model.train(); optimizer.zero_grad(set_to_none=True)
        warmup = epoch <= int(warmup_epochs)

        if warmup:
            xi, log_q = model.sample_posterior(int(R_train))
            elbo = model.log_likelihood(xi, force_all_on=True) + model.log_prior(xi) - log_q
        else:
            elbo = model.elbo_draws(int(R_train))["elbo"]

        (-elbo.mean()).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), float(grad_clip))
        optimizer.step()

        if epoch == 1 or epoch % int(eval_every) == 0 or epoch == int(epochs):
            model.eval()
            with torch.random.fork_rng(devices=[DEVICE] if DEVICE.type == "cuda" else []), torch.no_grad():
                xi_eval, _ = model.sample_posterior(int(R_eval))
                pred = model.decoder(X_eval, xi_eval, force_all_on=warmup)
                fm = metric.function_metrics(signal_eval, pred)
            print(f"epoch={epoch:04d} phase={'repr' if warmup else 'select':6s} "
                  f"diagMSE={fm['mse']:.5f} diagR2={fm['r2']:.4f}")

    if DEVICE.type == "cuda": torch.cuda.synchronize()
    train_time = time.perf_counter() - started

    model.eval()
    with torch.no_grad():
        xi_final, _ = model.sample_posterior(int(R_final))
        pred_test = model.decoder(X_test, xi_final)
        fm = metric.function_metrics(signal_test, pred_test)

    result = {"mse": float(fm["mse"]), "r2": float(fm["r2"]), "train_time_sec": float(train_time)}
    feature_pip = unit_pip = None

    if model.decoder.has_feature_gates:
        with torch.no_grad():
            feature_pip = model.decoder.feature_semantics(xi_final)["active"].float().mean(0).cpu().numpy()

        target = np.asarray(truth["feature_true"], dtype=float).reshape(-1) > 0.5
        selected = feature_pip > float(support_threshold)
        active, inactive = target, ~target
        brier_a = np.mean((1.0 - feature_pip[active]) ** 2)
        brier_0 = np.mean(feature_pip[inactive] ** 2) if inactive.any() else np.nan

        result.update({
            "tpr": float(selected[active].mean()),
            "auroc": float(roc_auc_score(target.astype(int), feature_pip)) if np.unique(target).size == 2 else np.nan,
            "brier_bal": float(0.5 * (brier_a + brier_0)) if inactive.any() else float(brier_a),
            "expected_support": float(feature_pip.sum()),
            "selected_support": int(selected.sum()),
            "mean_active_pip": float(feature_pip[active].mean()),
            "mean_inactive_pip": float(feature_pip[inactive].mean()) if inactive.any() else np.nan,
        })

    if model.decoder.has_unit_gates:
        with torch.no_grad():
            unit_pip = model.decoder.unit_semantics(xi_final)["active"].float().mean(0).cpu().numpy()

        expected_units = float(unit_pip.sum())
        result.update({
            "expected_active_units": expected_units,
            "selected_active_units": int((unit_pip > float(support_threshold)).sum()),
        })
        if len(tuple(hidden_dims)) == 1:
            result["unit_count_error"] = abs(expected_units - int(truth["n_true_units"]))

    if selection_mode == "feature_unit_induced_edge":
        result["network_density"] = float(metric.network_density(model.decoder, xi_final))
        result["path_density"] = float(metric.active_path_density(model.decoder, xi_final))

    return {"result": result, "model": model, "xi": xi_final.detach(),
            "feature_pip": feature_pip, "unit_pip": unit_pip}

## One experiment
Change the call below to choose the architecture and simulation. `slab_init` accepts `auto`, `legacy`, or `fan_in`. `fan_in` changes initial uncertainty, not the prior or the neural architecture.


In [40]:
def run_experiment(
    *, n=600, p=100, n_active=10, n_true_units=10,
    activation="relu", structure="mixed", features_per_unit=3,
    n_interactions=0, n_quadratic=0, extra_scale=0.5,
    target_signal_sd=1.5, sigma2=1.0, x_low=-2.5, x_high=2.5, data_seed=400,
    hidden_dims=(20,), selection_mode="feature_group",
    K_flow=4, flow_hidden_units=128, flow_hidden_layers=2, scale_clip=2.0,
    iaf_ordering_scheme="cyclic3", iaf_shuffle_within_role=True,
    gate_type="normalized_requ", gate_scale=1.0,
    train_frac=0.8, diagnostic_frac=0.10, epochs=1200, warmup_epochs=300,
    lr=3e-4, R_train=32, R_eval=128, R_final=500, eval_every=300,
    init_sd=0.5, init_loc_jitter=0.05, grad_clip=5.0,
    support_threshold=0.5, fit_seed=None,
    slab_init="auto", slab_sd_ratio=0.1, slab_bias_sd=0.02,
):
    if fit_seed is None:
        fit_seed = int(data_seed + 100_000 + p + 17 * len(tuple(hidden_dims)))

    X, y, feature_true, signal, info = simfun(
        n=n, p=p, n_active=n_active, n_true_units=n_true_units,
        activation=activation, structure=structure, features_per_unit=features_per_unit,
        n_interactions=n_interactions, n_quadratic=n_quadratic, extra_scale=extra_scale,
        sigma2=sigma2, target_signal_sd=target_signal_sd, x_low=x_low, x_high=x_high,
        seed=data_seed, device=DEVICE, dtype=DTYPE,
    )

    rng = np.random.default_rng(int(data_seed + 123_456))
    idx = rng.permutation(int(n))
    n_train = int(round(float(train_frac) * int(n)))
    train_idx, test_idx = idx[:n_train], idx[n_train:]
    n_diag = max(1, int(round(float(diagnostic_frac) * n_train)))
    diag_idx = train_idx[:n_diag]

    train_idx = torch.as_tensor(train_idx, device=DEVICE, dtype=torch.long)
    test_idx = torch.as_tensor(test_idx, device=DEVICE, dtype=torch.long)
    diag_idx = torch.as_tensor(diag_idx, device=DEVICE, dtype=torch.long)

    print(
        f"\nExperiment\n"
        f"  n/p/active     : {n}/{p}/{n_active}\n"
        f"  teacher        : {activation} | {structure} | {n_true_units} units"
        + (f" | {features_per_unit} features/unit" if structure == "mixed" else "") + "\n"
        f"  extra terms    : interaction={n_interactions} | quadratic={n_quadratic}\n"
        f"  fitted BNN     : {tuple(hidden_dims)} | {selection_mode}\n"
        f"  flow           : K={K_flow} | {iaf_ordering_scheme}\n"
        f"  training       : epochs={epochs} | warmup={warmup_epochs} | "
        f"R={R_train}/{R_eval}/{R_final}\n"
        f"  seed           : data={data_seed} | fit={fit_seed}"
    )

    out = train_fast(
        X[train_idx], y[train_idx], X[diag_idx], signal[diag_idx], X[test_idx], signal[test_idx],
        truth=info, selection_mode=selection_mode, hidden_dims=tuple(hidden_dims), sigma2=sigma2,
        init_sd=init_sd, K_flow=K_flow, flow_hidden_units=flow_hidden_units,
        flow_hidden_layers=flow_hidden_layers, scale_clip=scale_clip, flow_seed=int(fit_seed + 17),
        iaf_ordering_scheme=iaf_ordering_scheme, iaf_shuffle_within_role=iaf_shuffle_within_role,
        gate_type=gate_type, gate_scale=gate_scale, epochs=epochs, warmup_epochs=warmup_epochs,
        lr=lr, R_train=R_train, R_eval=R_eval, R_final=R_final, eval_every=eval_every,
        init_loc_jitter=init_loc_jitter, grad_clip=grad_clip,
        support_threshold=support_threshold, seed=fit_seed,
        slab_init=slab_init, slab_sd_ratio=slab_sd_ratio, slab_bias_sd=slab_bias_sd,
    )

    result = {
        "activation": activation, "structure": structure,
        "n_interactions": int(n_interactions), "n_quadratic": int(n_quadratic),
        **out["result"],
    }
    print("\nFinal result")
    print(result)

    out["result"] = result
    out["data_info"] = info
    return out


print("simfun, train_fast, run_experiment ready")

simfun, train_fast, run_experiment ready


## Predictor-level test


In [ ]:
exp = run_experiment(
    n=600, p=500, n_active=10, n_true_units=4,
    activation="relu", structure="mixed", features_per_unit=3,
    n_interactions=2, n_quadratic=0,
    hidden_dims=(80,), selection_mode="feature_group",
    K_flow=4, epochs=1200, warmup_epochs=300, data_seed=400,
)

In [ ]:
exp_unit = run_experiment(
    n=600, p=10, n_active=10, n_true_units=4,
    activation="relu", structure="mixed", features_per_unit=3,
    n_interactions=0, n_quadratic=0,
    hidden_dims=(12,), selection_mode="unit_group",
    K_flow=4, epochs=1200, warmup_epochs=300,
    data_seed=400,
)

In [48]:
exp_mlp = run_experiment(
    n=600, p=100, n_active=10, n_true_units=4,
    activation="relu", structure="mixed", features_per_unit=3,
    n_interactions=2, n_quadratic=1,
    hidden_dims=(20, 20), selection_mode="feature_unit_induced_edge",
    K_flow=4, epochs=4000, warmup_epochs=500, data_seed=400,
    slab_init="auto",
)



Experiment
  n/p/active     : 600/100/10
  teacher        : relu | mixed | 4 units | 3 features/unit
  extra terms    : interaction=2 | quadratic=1
  fitted BNN     : (20, 20) | feature_unit_induced_edge
  flow           : K=4 | cyclic3
  training       : epochs=4000 | warmup=500 | R=32/128/500
  seed           : data=400 | fit=100534
epoch=0001 phase=repr   diagMSE=1.92091 diagR2=-0.1700
epoch=0300 phase=repr   diagMSE=0.61370 diagR2=0.6262
epoch=0600 phase=select diagMSE=0.67452 diagR2=0.5892
epoch=0900 phase=select diagMSE=0.44351 diagR2=0.7299
epoch=1200 phase=select diagMSE=0.42701 diagR2=0.7399
epoch=1500 phase=select diagMSE=0.50943 diagR2=0.6897
epoch=1800 phase=select diagMSE=0.41848 diagR2=0.7451
epoch=2100 phase=select diagMSE=0.42231 diagR2=0.7428
epoch=2400 phase=select diagMSE=0.43586 diagR2=0.7345
epoch=2700 phase=select diagMSE=0.46269 diagR2=0.7182
epoch=3000 phase=select diagMSE=0.44030 diagR2=0.7318
epoch=3300 phase=select diagMSE=0.43137 diagR2=0.7373
epoch=3600 ph